# 01 - Ingesta AIS desde NOAA hacia Unity Catalog Volume

**Objetivo.** Descargar los ZIP oficiales AIS, comprobar su integridad, extraer los CSV en un Volume y leerlos con un esquema explícito.

**Entradas.** URLs oficiales del enunciado, catálogo configurado por `00_configuracion_unity_catalog` y el Volume `landing.raw_ais`.

**Salidas.** ZIP en `downloads/`, CSV en `extracted/` dentro del Volume, y métricas de descarga, extracción y lectura.

**Prerrequisitos.** Ejecutar antes `00_configuracion_unity_catalog`. La primera corrida se limita a 01-jun-2023.

**Validación.** Confirmar `INGESTION_OK`, `spark.version`, tiempos y conteo antes de habilitar los siete días.

In [0]:
import hashlib
import shutil
import time
import urllib.request
import zipfile
from pathlib import Path

from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

In [0]:
# Identificador provisional neutral; debe coincidir con el notebook 00.
TEAM_ID = "g06"
if not TEAM_ID.replace("_", "").isalnum() or TEAM_ID != TEAM_ID.lower():
    raise ValueError("TEAM_ID debe usar solo minúsculas, números y guiones bajos")

CATALOG = f"oceanwatch_{TEAM_ID}"
VOLUME_ROOT = Path(f"/Volumes/{CATALOG}/landing/raw_ais")

# 01-jun ya fue validado end-to-end contra el conteo local; ejecutar los siete días.
DATES_TO_INGEST = [
    "2023-06-01", "2023-06-02", "2023-06-03", "2023-06-04","2023-06-05", "2023-06-06", "2023-06-07",
]

OFFICIAL_URLS = {
    "2023-06-01": "https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_01.zip",
    "2023-06-02": "https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_02.zip",
    "2023-06-03": "https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_03.zip",
    "2023-06-04": "https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_04.zip",
    "2023-06-05": "https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_05.zip",
    "2023-06-06": "https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_06.zip",
    "2023-06-07": "https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_07.zip",
}

# No hay checksums oficiales publicados en el enunciado; el notebook registra SHA-256 observado.
EXPECTED_SHA256 = {}
RETRIES = 3
TIMEOUT_SECONDS = 120

In [0]:
# El esquema se declara explícitamente; MMSI se conserva como identificador de texto.
AIS_SCHEMA = StructType(
    [
        StructField("MMSI", StringType(), True),
        StructField("BaseDateTime", StringType(), True),
        StructField("LAT", DoubleType(), True),
        StructField("LON", DoubleType(), True),
        StructField("SOG", DoubleType(), True),
        StructField("COG", DoubleType(), True),
        StructField("Heading", DoubleType(), True),
        StructField("VesselName", StringType(), True),
        StructField("IMO", StringType(), True),
        StructField("CallSign", StringType(), True),
        StructField("VesselType", IntegerType(), True),
        StructField("Status", IntegerType(), True),
        StructField("Length", DoubleType(), True),
        StructField("Width", DoubleType(), True),
        StructField("Draft", DoubleType(), True),
        StructField("Cargo", IntegerType(), True),
        StructField("TransceiverClass", StringType(), True),
    ]
)


def sha256_file(path: Path, block_size: int = 1_048_576) -> str:
    """Calcula SHA-256 por bloques, apto para archivos grandes en el Volume."""
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(block_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_file(path: Path, expected_sha256: str | None = None) -> dict:
    """Verifica presencia, tamaño no nulo y checksum cuando exista una referencia oficial."""
    if not path.is_file():
        raise FileNotFoundError(f"Archivo no encontrado: {path}")
    size_bytes = path.stat().st_size
    if size_bytes == 0:
        raise ValueError(f"Archivo vacío: {path}")
    checksum = sha256_file(path)
    if expected_sha256 and checksum.lower() != expected_sha256.lower():
        raise ValueError(f"SHA-256 inesperado para {path.name}")
    return {"path": str(path), "bytes": size_bytes, "sha256": checksum}


def download_with_retries(url: str, destination: Path, expected_sha256: str | None) -> tuple[dict, float]:
    """Descarga a un archivo temporal y lo mueve solo después de recibir HTTP 200."""
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".part")
    last_error = None
    started = time.perf_counter()

    for attempt in range(1, RETRIES + 1):
        try:
            with urllib.request.urlopen(url, timeout=TIMEOUT_SECONDS) as response, temporary.open("wb") as output:
                if response.status != 200:
                    raise OSError(f"HTTP inesperado: {response.status}")
                shutil.copyfileobj(response, output, length=1_048_576)
            temporary.replace(destination)
            return verify_file(destination, expected_sha256), time.perf_counter() - started
        except Exception as error:
            last_error = error
            temporary.unlink(missing_ok=True)
            if attempt < RETRIES:
                time.sleep(2 ** (attempt - 1))

    raise RuntimeError(f"Descarga fallida tras {RETRIES} intentos: {url}") from last_error


def extract_single_csv(archive: Path, destination_dir: Path) -> tuple[Path, float]:
    """Extrae exactamente un CSV y elimina componentes de ruta no confiables del ZIP."""
    if not zipfile.is_zipfile(archive):
        raise ValueError(f"ZIP inválido: {archive}")
    started = time.perf_counter()

    with zipfile.ZipFile(archive) as zipped:
        csv_members = [member for member in zipped.infolist() if not member.is_dir() and member.filename.lower().endswith(".csv")]
        if len(csv_members) != 1:
            raise ValueError(f"Se esperaba exactamente un CSV en {archive.name}; encontrados: {len(csv_members)}")
        member = csv_members[0]
        destination = destination_dir / Path(member.filename).name
        destination.parent.mkdir(parents=True, exist_ok=True)
        with zipped.open(member) as source, destination.open("wb") as target:
            shutil.copyfileobj(source, target, length=1_048_576)

    verify_file(destination)
    return destination, time.perf_counter() - started


def read_ais_csv(csv_path: Path):
    """Lee con esquema y valida que el encabezado del CSV coincida con las 17 columnas esperadas."""
    return (
        spark.read.option("header", True)
        .option("enforceSchema", "false")
        .option("mode", "FAILFAST")
        .schema(AIS_SCHEMA)
        .csv(str(csv_path))
    )


def union_daily_frames(daily_frames):
    """Unión preparada para siete días; no invocar hasta aprobar la prueba de un día."""
    unioned = daily_frames[0]
    for frame in daily_frames[1:]:
        unioned = unioned.unionByName(frame, allowMissingColumns=False)
    return unioned

## Ingesta controlada: únicamente 01-jun-2023

`DATES_TO_INGEST` se amplía a los siete días después de revisar esta salida. La función de unión queda lista, pero no se invoca en esta primera corrida.

In [0]:
#Celda de limpieza
import shutil

downloads_dir = VOLUME_ROOT / "downloads"
extracted_dir = VOLUME_ROOT / "extracted"

if downloads_dir.exists():
    shutil.rmtree(downloads_dir)

if extracted_dir.exists():
    shutil.rmtree(extracted_dir)

downloads_dir.mkdir(parents=True, exist_ok=True)
extracted_dir.mkdir(parents=True, exist_ok=True)

print("Entorno de ingestión reiniciado")

Entorno de ingestión reiniciado


In [0]:
daily_frames = []
ingestion_metrics = []

for day in DATES_TO_INGEST:
    if day not in OFFICIAL_URLS:
        raise KeyError(f"Fecha sin URL oficial: {day}")

    archive_path = VOLUME_ROOT / "downloads" / f"AIS_{day.replace('-', '_')}.zip"
    csv_dir = VOLUME_ROOT / "extracted" / f"ingestion_date={day}"
    if archive_path.exists():
        if not zipfile.is_zipfile(archive_path):
            print(f"ZIP corrupto detectado. Eliminando {archive_path.name}")
            archive_path.unlink()

    if not archive_path.exists():
         download_info, download_seconds = download_with_retries(
            OFFICIAL_URLS[day],
            archive_path,
            EXPECTED_SHA256.get(day)
        )
         if not zipfile.is_zipfile(archive_path):
            archive_path.unlink(missing_ok=True)
            raise RuntimeError(
                f"La descarga de {archive_path.name} produjo un archivo inválido."
            )
    else:
        download_info = verify_file(archive_path)
        download_seconds = 0

    csv_path = csv_dir / f"AIS_{day.replace('-', '_')}.csv"

    if not csv_path.exists():
        csv_path, extract_seconds = extract_single_csv(
            archive_path, csv_dir
        )
    else:
        print(f"Usando CSV existente: {csv_path.name}")
        extract_seconds = 0

    read_started = time.perf_counter()
    daily_frame = read_ais_csv(csv_path)
    rows = daily_frame.count()
    read_count_seconds = time.perf_counter() - read_started
    if daily_frame.schema != AIS_SCHEMA:
        raise RuntimeError(f"Esquema inesperado en {day}")

    daily_frames.append(daily_frame)
    ingestion_metrics.append(
        {
            "day": day,
            "archive_bytes": download_info["bytes"],
            "archive_sha256": download_info["sha256"],
            "csv_path": str(csv_path),
            "download_seconds": round(download_seconds, 2),
            "extract_seconds": round(extract_seconds, 2),
            "rows": rows,
            "read_count_seconds": round(read_count_seconds, 2),
        }
    )

display(spark.createDataFrame(ingestion_metrics))
print("INGESTION_OK")
print(f"spark_version={spark.version}")
print(f"volume_root={VOLUME_ROOT}")
print(f"frames_prepared={len(daily_frames)}")

archive_bytes,archive_sha256,csv_path,day,download_seconds,extract_seconds,read_count_seconds,rows
343949892,e6287c47f9659fd3bc6c0456579e93f07e3ec099c347514757a86413d0c99d6a,/Volumes/oceanwatch_g06/landing/raw_ais/extracted/ingestion_date=2023-06-01/AIS_2023_06_01.csv,2023-06-01,16.62,7.36,3.81,8808904
354145683,9f01ff4c3a21240c41ed0ca64a945be7fb8bee0f5bff049dc5c3e727aa02b8ac,/Volumes/oceanwatch_g06/landing/raw_ais/extracted/ingestion_date=2023-06-02/AIS_2023_06_02.csv,2023-06-02,18.32,7.35,2.16,9052241
312248025,070ccec839de290cba810308753b56d2bac6839a9fd076df6ab089b525e046c8,/Volumes/oceanwatch_g06/landing/raw_ais/extracted/ingestion_date=2023-06-03/AIS_2023_06_03.csv,2023-06-03,25.74,7.28,2.1,8036348
332457486,396efd491ea992aded336ba85bfed1cd8079baf294fb1fadf215e69d40ed42a0,/Volumes/oceanwatch_g06/landing/raw_ais/extracted/ingestion_date=2023-06-04/AIS_2023_06_04.csv,2023-06-04,18.91,7.33,1.96,8522645
335658758,00899e6450a4cc3170059ed990c94e060257e86f22edd5c6683f282e5ce7f4c8,/Volumes/oceanwatch_g06/landing/raw_ais/extracted/ingestion_date=2023-06-05/AIS_2023_06_05.csv,2023-06-05,13.81,7.37,2.73,8613757
335300028,c8b4acd430a675fbffe3665d9ed9843ce0696c947e4d35411a504ad2ffc595f4,/Volumes/oceanwatch_g06/landing/raw_ais/extracted/ingestion_date=2023-06-06/AIS_2023_06_06.csv,2023-06-06,44.88,7.32,3.25,8587938
347918296,03b9b5ddb1247f08a4c3bab738c7b6c569c93094f82e8830cfea4853f9a5bed7,/Volumes/oceanwatch_g06/landing/raw_ais/extracted/ingestion_date=2023-06-07/AIS_2023_06_07.csv,2023-06-07,14.87,8.02,2.05,8911726


INGESTION_OK
spark_version=4.2.0
volume_root=/Volumes/oceanwatch_g06/landing/raw_ais
frames_prepared=7


In [0]:
# La unión exige el mismo contrato de columnas en los siete DataFrames.
union_started = time.perf_counter()
all_days = union_daily_frames(daily_frames)
total_rows = all_days.count()
union_seconds = time.perf_counter() - union_started
expected_total_rows = 60_533_559
if total_rows != expected_total_rows:
    raise RuntimeError(f"Total inesperado: {total_rows:,} vs {expected_total_rows:,}")

display(spark.createDataFrame([(total_rows, expected_total_rows, total_rows - expected_total_rows, round(union_seconds, 2))],
    "filas_unidas long, filas_esperadas long, diferencia long, segundos_union double"))
print("UNION_OK")

filas_unidas,filas_esperadas,diferencia,segundos_union
60533559,60533559,0,15.81


UNION_OK
